# KG-GT Method 1 — Preprocess (all subjects/series) + Train Transformer Encoder

Pipeline (method1.tex §3.2-3.3):

1. **Preprocess** every subject × series HS file: EEG (BP 0.1-40 Hz → notch → ASR → CAR → delta-band), EMG (BP 30-300 → rectify → LP 10 Hz → decimate ×8 → z-score), kinematics raw → `k_t` (13-dim). Windowed (`W=500`, stride 50). Cached to `.npz`.
2. **CCA** fit on train: EEG 32 → 13 canonical components aligned with `k_t`.
3. **Transformer encoder** (L=4, H=8, d=256) over CCA-projected EEG + sinusoidal PE → `H_temp`.
4. **Readout head** Linear(256→5) — *temporary* stand-in for the not-yet-built kinematic-guided GAT + decoder. Lets us train the encoder end-to-end against EMG now.
5. **Train**: Adam 1e-3, MSE loss, grad-clip 1.0, early stopping.

> SoftDTW term (`src/losses/soft_dtw.py`) is still a stub → loss here is MSE only. GAT replaces the linear head later.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# project root on path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.dataset import WAYEEGDataset
from src.preprocessing.eeg import preprocess_eeg_from_config
from src.preprocessing.emg import preprocess_emg_from_config
from src.preprocessing.cca import make_cca_from_config
from src.models.transformer import build_transformer_from_config
from src.losses import build_loss_from_config
from src.training import TrainConfig, train_model, evaluate, save_checkpoint
from src.utils import get_device, set_seed

set_seed(42)
DEVICE = get_device()          # CUDA on the GTX 1660 Ti, else CPU
print('root :', ROOT)
print('torch:', torch.__version__, '| cuda?', torch.cuda.is_available(), '| device:', DEVICE)
if DEVICE.type == 'cuda':
    print('gpu  :', torch.cuda.get_device_name(0))

In [ ]:
cfg = yaml.safe_load(open(ROOT / 'configs' / 'default.yaml'))

DATA_DIR  = ROOT / cfg['data']['raw_dir']
CACHE_DIR = ROOT / cfg['data']['cache_dir']
W         = cfg['data']['window_size']
STRIDE    = cfg['data']['stride']
N_CCA     = cfg['preprocessing']['cca']['n_components']

# All 12 subjects per request. HS files are large (30-66 MB each) -> first build
# is slow and memory-heavy; .npz cache makes re-runs fast. Shrink for a quick
# smoke test, e.g. PARTICIPANTS = [1].
PARTICIPANTS = cfg['data']['participants']

print('data dir     :', DATA_DIR)
print('participants :', PARTICIPANTS)
print('window/stride:', W, '/', STRIDE, '| CCA comps:', N_CCA)

## 1. Preprocessing function

Applied per raw series inside the Dataset, before windowing. Handles **EEG** and **EMG** only — kinematics stay raw 36-col because `dataset.extract_kt()` builds `k_t` internally. EMG z-score is left to a second pass (needs global train statistics).

In [ ]:
def make_preprocess_fn(cfg):
    eeg_cfg = cfg['preprocessing']['eeg']
    emg_cfg = cfg['preprocessing']['emg']

    def preprocess_fn(s):
        s = dict(s)
        s['eeg'] = preprocess_eeg_from_config(s['eeg'], float(s['fs_eeg']), eeg_cfg)
        s['emg'] = preprocess_emg_from_config(s['emg'], float(s['fs_emg']), emg_cfg)
        # kin: leave raw (T, 36) -> dataset.extract_kt() handles k_t
        return s
    return preprocess_fn

preprocess_fn = make_preprocess_fn(cfg)

## 2. Build datasets (train / val / test)

Series-level split (method1.tex): train S1-7, val S8, test S9. First run preprocesses + caches; later runs hit the cache.

In [ ]:
def build_ds(split):
    t0 = time.time()
    ds = WAYEEGDataset(
        data_dir=DATA_DIR,
        participants=PARTICIPANTS,
        split=split,
        window_size=W,
        stride=STRIDE,
        preprocess_fn=preprocess_fn,
        cache_dir=CACHE_DIR,
    )
    print(f'{split:5s} | {len(ds):6d} windows | {time.time()-t0:6.1f}s')
    return ds

train_ds = build_ds('train')
val_ds   = build_ds('val')
test_ds  = build_ds('test')

eeg0, kin0, emg0 = train_ds[0]
print('sample shapes  eeg', tuple(eeg0.shape), 'kin', tuple(kin0.shape), 'emg', tuple(emg0.shape))

## 3. Fit CCA + EMG z-score on the training set

Each window tuple shares its parent series arrays by reference → dedup by `id()` so CCA / stats are fit on whole series once, not per overlapping window.

In [ ]:
def unique_series_arrays(ds):
    """Return (eeg_list, kin_list, emg_list) of unique series arrays in ds."""
    seen, eegs, kins, emgs = set(), [], [], []
    for eeg_all, kin_all, emg_all, _ in ds._windows:
        key = id(eeg_all)
        if key in seen:
            continue
        seen.add(key)
        eegs.append(eeg_all); kins.append(kin_all); emgs.append(emg_all)
    return eegs, kins, emgs

tr_eeg, tr_kin, tr_emg = unique_series_arrays(train_ds)
print('unique train series:', len(tr_eeg))

# --- CCA: EEG 32 -> N_CCA aligned with k_t ---
cca = make_cca_from_config(cfg['preprocessing']['cca'])
cca.fit(tr_eeg, tr_kin)
print('CCA fitted -> n_components =', cca.n_components)
# Export the fitted CCA as an on-device torch projector. This is the key GPU
# fix: the projection becomes a matmul on the GPU, so prepare_batch no longer
# round-trips every batch to the CPU for sklearn (the old bottleneck).
cca_gpu = cca.torch_projector(DEVICE)
print('TorchCCA on', cca_gpu.device, '| n_components =', cca_gpu.n_components)
# --- EMG z-score stats over all train series (per channel) ---
emg_concat = np.concatenate(tr_emg, axis=0)
EMG_MEAN = torch.tensor(emg_concat.mean(0), dtype=torch.float32, device=DEVICE)
EMG_STD  = torch.tensor(emg_concat.std(0) + 1e-8, dtype=torch.float32, device=DEVICE)
print('EMG mean', np.round(EMG_MEAN.cpu().numpy(), 3))
print('EMG std ', np.round(EMG_STD.cpu().numpy(), 3))

## 4. Batch transform: EEG → CCA, EMG → z-score

Applied on the fly. CCA projection runs in numpy (sklearn) so we move EEG to CPU for it, then back to device.

In [ ]:
def prepare_batch(eeg, emg):
    """eeg (B,W,32), emg (B,W,5) -> cca_eeg (B,W,N_CCA), emg_norm (B,W,5)."""
    B, Wn, _ = eeg.shape
    flat = eeg.reshape(-1, 32).to(DEVICE)             # (B*W, 32)
    proj = cca_gpu.transform(flat).reshape(B, Wn, N_CCA)  # (B,W,N_CCA)
    cca_eeg = proj
    emg_norm = (emg.to(DEVICE) - EMG_MEAN) / EMG_STD
    return cca_eeg, emg_norm

# sanity
_e, _k, _m = next(iter(DataLoader(train_ds, batch_size=4)))
_ce, _mn = prepare_batch(_e, _m)
print('cca_eeg', tuple(_ce.shape), '| emg_norm', tuple(_mn.shape), _mn.device)

## 5. Model: Transformer encoder + linear readout head

`H_temp (B,T,256)` → `Linear(256→5)` per time step. The head is a placeholder for the kinematic-guided GAT + decoder.

In [ ]:
class TransformerRegressor(nn.Module):
    def __init__(self, cfg, input_dim, out_channels=5):
        super().__init__()
        self.encoder = build_transformer_from_config(cfg['model']['transformer'], input_dim)
        self.head = nn.Linear(self.encoder.d_model, out_channels)

    def forward(self, x):
        h = self.encoder(x)      # (B,T,256)
        return self.head(h)      # (B,T,5)

model = TransformerRegressor(cfg, input_dim=N_CCA).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print('trainable params:', n_params)

## 6. Training

Adam (lr 1e-3, halve on val plateau), MSE loss, grad-clip 1.0, early stopping. `MAX_EPOCHS` kept small for a runnable demo; raise toward `cfg['training']['max_epochs']` (500) for a full run.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
tr_cfg = cfg['training']
BATCH = tr_cfg['batch_size']

# CUDA: overlap data loading with compute. CPU: keep it simple (workers add overhead).
loader_kw = dict(pin_memory=True, num_workers=2, persistent_workers=True)\
    if DEVICE.type == 'cuda' else dict()
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  drop_last=True, **loader_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, **loader_kw)
loss_fn = build_loss_from_config(tr_cfg).to(DEVICE)
# max_epochs kept small for the demo; pass max_epochs=None to use config (500).
train_config = TrainConfig.from_config(tr_cfg, max_epochs=15)
result = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    prepare_batch=prepare_batch,
    loss_fn=loss_fn,
    device=DEVICE,
    cfg=train_config,)
hist = result.history
print('best val loss:', round(result.best_val, 4))
# persist the best weights
ckpt_dir = ROOT / 'outputs'; ckpt_dir.mkdir(exist_ok=True)
save_checkpoint(str(ckpt_dir / 'transformer_best.pt'), model, train_config, result.best_val)

## 7. Loss curve

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(hist['train'])+1), hist['train'], 'o-', label='train')
plt.plot(range(1, len(hist['val'])+1),   hist['val'],   's-', label='val')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('Transformer training')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 8. Predicted vs target EMG on a test window

In [ ]:
EMG_NAMES = ['Ant. Deltoid', 'Ext. Carpi Rad.', 'Flex. Digitorum', 'Ext. Dig. Comm.', '1st Dors. Inteross.']

model.eval()
eeg, kin, emg = test_ds[len(test_ds)//2]
x, y = prepare_batch(eeg.unsqueeze(0), emg.unsqueeze(0))
with torch.no_grad():
    pred = model(x)[0].cpu().numpy()
y = y[0].cpu().numpy()

t = np.arange(W) / cfg['data']['fs_eeg']
fig, axes = plt.subplots(5, 1, figsize=(9, 10), sharex=True)
for c, ax in enumerate(axes):
    ax.plot(t, y[:, c],   'k-',  lw=1.4, label='target')
    ax.plot(t, pred[:, c], 'r--', lw=1.4, label='pred')
    ax.set_ylabel(EMG_NAMES[c], fontsize=8)
    ax.grid(alpha=0.3)
    if c == 0:
        ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('time (s)')
fig.suptitle('Predicted vs target EMG (z-scored) — test window')
plt.tight_layout(); plt.show()

## 9. Test metrics (per channel)

RMSE, MAE, Pearson r on z-scored EMG over the full test split.

In [ ]:
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, **loader_kw)
metrics = evaluate(
    model=model,
    loader=test_loader,
    prepare_batch=prepare_batch,
    device=DEVICE,
    channel_names=EMG_NAMES,
    n_channels=5,
)
print(metrics.as_table())